# SEA-LION Finetuned -- Prompt Variant Probe

Ad-hoc exploration notebook: loads the **finetuned** SEA-LION adapter
(`tdr-experiments/evals/sealion/adapter`, per `common.adapter_dir("sealion")`)
and runs it against 4 differently-phrased prompt styles, to see how the
model behaves when it is NOT given the exact MCQ-with-letters format
`run_eval.py` scores it on.

5 homographs, hardcoded from `data/benchmark.jsonl` (kaya, tayo, nasa,
dating, paano) -- picked once and pinned here so this notebook is
self-contained and its inputs never silently drift if `benchmark.jsonl`
changes.

4 prompt variants x 5 homographs = 20 total generations:
1. `mcq_lettered` -- the exact format `run_eval.py` scores against
   (A/B/C/D choices, answer = a single letter).
2. `mcq_unlettered` -- same 4 choices, no letters; asks the model to
   output the correct variant string itself.
3. `open_no_choices` -- the question alone, no choices given at all.
4. `rewrite_instruction` -- an instruction-style rewrite prompt, closer
   in shape to what `finetune.py` actually trained on (instruction +
   sentence -> rewritten sentence) than the three MCQ-flavored formats
   above.

Assumes `evals/sealion/adapter/` already exists, i.e. `python
finetune.py --model sealion` has already been run -- this notebook does
not train anything, it only loads the saved adapter and probes it.

## Setup

Path resolution mirrors the other notebooks in this project (`fewshot_prep.ipynb`,
`dataprep.ipynb`): works whether the kernel's working directory is the
project root or `tdr-experiments/` itself.

In [ ]:
import sys
from pathlib import Path

_candidates = [Path("tdr-experiments"), Path(".")]
PROJECT_DIR = next((p for p in _candidates if (p / "common.py").is_file()), None)
assert PROJECT_DIR is not None, f"tdr-experiments (with common.py) not found in any of: {_candidates}"
sys.path.insert(0, str(PROJECT_DIR.resolve()))

print(f"PROJECT_DIR: {PROJECT_DIR.resolve()}")

## Load the finetuned SEA-LION adapter

Same loading path `run_eval.py` uses for its `finetuned` mode:
`load_model_and_tokenizer("sealion", adapter=True, for_training=False)`.

In [ ]:
import torch

from common import adapter_dir, build_chat_prompt, load_model_and_tokenizer, parse_response

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

ADAPTER_PATH = adapter_dir("sealion")
assert ADAPTER_PATH.exists(), (
    f"Adapter not found at {ADAPTER_PATH} -- run `python finetune.py --model sealion` first."
)

model, tokenizer = load_model_and_tokenizer("sealion", adapter=True, for_training=False)
tokenizer.padding_side = "left"
model.eval()

print(f"Loaded finetuned sealion adapter from {ADAPTER_PATH}")

## 5 hardcoded samples (different homographs)

Pulled verbatim from `data/benchmark.jsonl` -- `kaya`, `tayo`, and `nasa`
as given, plus `dating` and `paano` (2 more distinct homographs, picked
from the same file).

In [ ]:
SAMPLES = [
    {
        "id": "raw_balitanlp_test_00009.txt_5_kaya_1",
        "homograph": "kaya",
        "prompt": "How should the word kaya be written with the correct diacritic marks in the sentence \"Hindi kayang bumili  ng mga kababayan nating mahihirap ng LPG kaya uling o gaas ang kanilang ginagamit sa panluto\"?",
        "choices": ["kayà", "kayâ", "kaya", "káya"],
        "label": 1,
        "gold_variant": "kayâ",
        "asymmetry_type": "Extreme Asymmetry",
    },
    {
        "id": "raw_balitanlp_test_00010.txt_3_tayo_1",
        "homograph": "tayo",
        "prompt": "How should the word tayo be written with the correct diacritic marks in the sentence \"Pero dahil tayo ay nasa kasagsagan na ng makabagong panahon isang post mo lamang ng kakaibang video, nakakatuwa man o nakakainis agad na itong magba-viral at puputaktihin na ng mga netizens lalo na ng mga chismosa nating kapitbahay\"?",
        "choices": ["tâyo", "tayô", "tayo", "táyo"],
        "label": 3,
        "gold_variant": "táyo",
        "asymmetry_type": "Extreme Asymmetry",
    },
    {
        "id": "raw_balitanlp_test_00010.txt_3_nasa_1",
        "homograph": "nasa",
        "prompt": "How should the word nasa be written with the correct diacritic marks in the sentence \"Pero dahil tayo ay nasa kasagsagan na ng makabagong panahon isang post mo lamang ng kakaibang video, nakakatuwa man o nakakainis agad na itong magba-viral at puputaktihin na ng mga netizens lalo na ng mga chismosa nating kapitbahay\"?",
        "choices": ["nasà", "nasa", "nàsa", "nása"],
        "label": 3,
        "gold_variant": "nása",
        "asymmetry_type": "Extreme Asymmetry",
    },
    {
        "id": "raw_balitanlp_test_00009.txt_8_dating_1",
        "homograph": "dating",
        "prompt": "How should the word dating be written with the correct diacritic marks in the sentence \"Ayon sa mambabatas, isang supot ngayon ng uling ay nagkakahalaga ng P15 mula sa dating presyo nitong P10\"?",
        "choices": ["datìng", "datíng", "dating", "dáting"],
        "label": 3,
        "gold_variant": "dáting",
        "asymmetry_type": "Moderate Asymmetry",
    },
    {
        "id": "raw_balitanlp_test_00009.txt_16_paano_1",
        "homograph": "paano",
        "prompt": "How should the word paano be written with the correct diacritic marks in the sentence \"\u201cMaawa naman sila, paano pa ipagdiriwang ng mga mahihirap ang darating na Kapaskuhan\"?",
        "choices": ["paanô", "paano", "paanó", "paáno"],
        "label": 3,
        "gold_variant": "paáno",
        "asymmetry_type": "Extreme Asymmetry",
    },
]

assert len({s["homograph"] for s in SAMPLES}) == 5, "SAMPLES must cover 5 distinct homographs"
print(f"Loaded {len(SAMPLES)} hardcoded sample(s): {[s['homograph'] for s in SAMPLES]}")

## 4 prompt variants

`extract_sentence` recovers the bare sentence embedded inside a
`benchmark.jsonl` `prompt` string -- copied verbatim from
`run_eval.py`'s `_extract_sentence_from_prompt` (same recovery logic,
kept local rather than importing that module's private helper).

Each variant gets its own system prompt (the output format each one
expects is different, so "answer with a single letter" only makes sense
for `mcq_lettered`) and its own `max_new_tokens` budget (a single letter
needs far fewer tokens than a full rewritten sentence).

In [ ]:
import re


def extract_sentence(prompt: str) -> str:
    match = re.search(r'in the sentence "(?P<sentence>.*)"\?$', prompt, re.DOTALL)
    assert match, f"Could not extract sentence from prompt: {prompt!r}"
    return match.group("sentence")


SYSTEM_PROMPTS = {
    "mcq_lettered": (
        "You are a Tagalog diacritic restoration assistant. Answer using "
        "ONLY a single letter: A, B, C, or D. Do not include any explanation, "
        "reasoning, punctuation, or extra text -- output ONLY the letter."
    ),
    "mcq_unlettered": (
        "You are a Tagalog diacritic restoration assistant. From the list of "
        "variants given, respond with ONLY the correct variant, copied exactly "
        "as written (including its diacritic marks). Do not include any "
        "explanation, reasoning, punctuation, or extra text."
    ),
    "open_no_choices": (
        "You are a Tagalog diacritic restoration assistant. Respond with ONLY "
        "the correctly diacritized form of the target word, exactly as it "
        "should appear in the sentence. Do not include any explanation, "
        "reasoning, punctuation, or extra text."
    ),
    "rewrite_instruction": (
        "You are a Tagalog diacritic restoration assistant. Respond with ONLY "
        "the rewritten sentence. Do not include any explanation, reasoning, or "
        "extra text."
    ),
}

MAX_NEW_TOKENS = {
    "mcq_lettered": 8,           # matches common.MAX_NEW_TOKENS / run_eval.py
    "mcq_unlettered": 16,        # one diacritized word
    "open_no_choices": 16,       # one diacritized word
    "rewrite_instruction": 100,  # full sentence echoed back
}

VARIANTS = ["mcq_lettered", "mcq_unlettered", "open_no_choices", "rewrite_instruction"]


def build_user_message(variant: str, record: dict, sentence: str) -> str:
    homograph = record["homograph"]
    choices = record["choices"]
    question = (
        f'How should the word {homograph} be written with the correct diacritic '
        f'marks in the sentence "{sentence}"?'
    )

    if variant == "mcq_lettered":
        letters = ["A", "B", "C", "D"]
        lines = [question, ""] + [f"{letter}. {choice}" for letter, choice in zip(letters, choices)]
        return "\n".join(lines)

    if variant == "mcq_unlettered":
        lines = [question, ""] + [f'"{choice}"' for choice in choices]
        return "\n".join(lines)

    if variant == "open_no_choices":
        return question

    if variant == "rewrite_instruction":
        return (
            f'Modify the following Tagalog sentence by adding a diacritic in '
            f'the word {homograph}: "{sentence}"?'
        )

    raise ValueError(f"Unknown variant: {variant}")

## Build the 20 (homograph, variant) prompts

In [ ]:
runs = []
for record in SAMPLES:
    sentence = extract_sentence(record["prompt"])
    for variant in VARIANTS:
        runs.append({
            "id": record["id"],
            "homograph": record["homograph"],
            "label": record["label"],
            "gold_variant": record["gold_variant"],
            "variant": variant,
            "user_message": build_user_message(variant, record, sentence),
        })

assert len(runs) == 20, f"Expected 20 runs (5 homographs x 4 variants), got {len(runs)}"
print(f"Built {len(runs)} prompt(s).")

## Generate

One prompt at a time (batch size 1) rather than `run_eval.py`'s batched
path -- this is for eyeballing individual outputs, not throughput.
Greedy decoding (`do_sample=False`), same as `run_eval.py`.

In [ ]:
for run in runs:
    system_prompt = SYSTEM_PROMPTS[run["variant"]]
    prompt_text = build_chat_prompt(tokenizer, run["user_message"], system_prompt=system_prompt)

    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS[run["variant"]],
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )

    input_len = inputs["input_ids"].shape[1]
    decoded = tokenizer.decode(output_ids[0][input_len:], skip_special_tokens=True)
    run["raw_output"] = decoded

    print(f"[{run['homograph']:8s} / {run['variant']:20s}] gold={run['gold_variant']!r}  ->  {decoded!r}")

## Results

`correct` is only a meaningful accuracy signal for `mcq_lettered`, where
it re-uses `common.parse_response` (the same letter-parsing `run_eval.py`
scores with). For the other 3 variants there's no fixed output contract
to parse against, so `correct` there just checks whether `gold_variant`
appears verbatim in the raw output -- a rough signal, not a real score
(e.g. it would miss a correct answer wrapped in different punctuation).

In [ ]:
import pandas as pd


def score(run: dict) -> bool | None:
    if run["variant"] == "mcq_lettered":
        parsed = parse_response(run["raw_output"])
        if parsed is None:
            return None  # parse error, same as run_eval.py
        return ord(parsed["answer"]) - ord("a") == run["label"]
    return run["gold_variant"] in run["raw_output"]


for run in runs:
    run["correct"] = score(run)

pd.set_option("display.max_colwidth", 60)
df_runs = pd.DataFrame(runs)
df_runs[["homograph", "variant", "gold_variant", "raw_output", "correct"]]